In [2]:
pip install fredapi python-dotenv pandas duckdb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import duckdb

con = duckdb.connect("titan_corevault.duckdb")
%sql con

UsageError: Line magic function `%sql` not found.


In [35]:
!pip install jupysql duckdb-engine -q

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False

%sql duckdb:///titan_corevault.duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False

%sql duckdb:///titan_corevault.duckdb

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


RuntimeError: (_duckdb.ConnectionException) Connection Error: Can't open a connection to same database file with a different configuration than existing connections
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [3]:
%%writefile .env
FRED_API_KEY=dcf5dc511af8d9e8f55bd016d5b35aaa

Writing .env


In [8]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred

load_dotenv()
api_key = os.getenv("FRED_API_KEY")

fred = Fred(api_key=api_key)
sofr_series = fred.get_series('SOFR')

df_sofr = sofr_series.to_frame(name='sofr_rate').reset_index()
df_sofr.rename(columns={'index': 'observation_date'}, inplace=True)
df_sofr.dropna(inplace=True)

con = duckdb.connect('titan_corevault.duckdb')

con.execute("""
    CREATE TABLE IF NOT EXISTS fact_sofr_rates (
        observation_date DATE PRIMARY KEY,
        sofr_rate DOUBLE
    )
""")

con.execute("INSERT OR REPLACE INTO fact_sofr_rates SELECT * FROM df_sofr")

con.execute("SELECT * FROM fact_sofr_rates ORDER BY observation_date DESC LIMIT 5").df()

,observation_date,sofr_rate
0,2026-08-14,3.62
1,2026-08-13,3.62
2,2026-08-12,3.62
3,2026-08-11,3.64
4,2026-08-10,3.63


In [9]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.2 MB/s  0:00:00


In [12]:
import duckdb
import pandas as pd
from faker import Faker
import random
import uuid

fake = Faker('en_US')
Faker.seed(42)

print("Generating synthetic Titan banking users...")

users=[]
for _ in range(100):
    users.append({
        "user_id": str(uuid.uuid4()),
        "full_name": fake.name(),
        "email": fake.email(),
        "ssn_hashed": fake.sha256(),
        "credit_score": random.randint(580, 850),
        "created_at": fake.date_between(start_date='-2y', end_date='today')
    })

df_users = pd.DataFrame(users)

accounts = []
transactions = []

account_types = ['High-Yield Savings', 'Checking', 'Credit Line']

for user in users:
    num_accounts = random.choice([1, 2])
    for _ in range(num_accounts):
        account_id = str(uuid.uuid4())
        acct_type = random.choice(account_types)
        
        accounts.append({
            "account_id": account_id,
            "user_id": user["user_id"],
            "account_type": acct_type,
            "current_balance": round(random.uniform(500.0, 25000.0), 2),
            "status": "Active"
        })

for _ in range(random.randint(5, 10)):
            transactions.append({
                "transaction_id": str(uuid.uuid4()),
                "account_id": account_id,
                "transaction_date": fake.date_between(start_date='-30d', end_date='today'),
                "amount": round(random.uniform(-500.0, 1500.0), 2),
                "merchant_category": random.choice(['Groceries', 'Payroll Direct Deposit', 'P2P Transfer', 'Utilities', 'Online Retail']),
                "status": "Settled"
            })

df_accounts = pd.DataFrame(accounts)
df_transactions = pd.DataFrame(transactions)

con = duckdb.connect('titan_corevault.duckdb')

con.execute("CREATE TABLE IF NOT EXISTS dim_users AS SELECT * FROM df_users")
con.execute("CREATE TABLE IF NOT EXISTS dim_accounts AS SELECT * FROM df_accounts")
con.execute("CREATE TABLE IF NOT EXISTS fact_transactions AS SELECT * FROM df_transactions")

print("Successfully populated Titan CoreVault relational tables!")
print(f"Users created: {len(df_users)}")
print(f"Accounts created: {len(df_accounts)}")
print(f"Transactions created: {len(df_transactions)}")

con.close()

Generating synthetic Titan banking users...
Successfully populated Titan CoreVault relational tables!
Users created: 100
Accounts created: 145
Transactions created: 5


In [24]:
import duckdb

con = duckdb.connect('titan_corevault.duckdb')

con.execute("SHOW TABLES").df()

,name
0,dim_accounts
1,dim_users
2,fact_sofr_rates
3,fact_transactions


In [25]:
import duckdb

con = duckdb.connect('titan_corevault.duckdb')

con.execute("DESCRIBE dim_users").df()

,column_name,column_type,null,key,default,extra
0,user_id,VARCHAR,YES,None,None,None
1,full_name,VARCHAR,YES,None,None,None
2,email,VARCHAR,YES,None,None,None
3,ssn_hashed,VARCHAR,YES,None,None,None
4,credit_score,BIGINT,YES,None,None,None
5,created_at,DATE,YES,None,None,None


In [21]:
print("=== USERS TABLE ===")
display(con.execute("SELECT * FROM dim_users LIMIT 5").df())

print("\n=== ACCOUNTS TABLE ===")
display(con.execute("SELECT * FROM dim_accounts LIMIT 5").df())

print("\n=== TRANSACTIONS TABLE ===")
display(con.execute("SELECT * FROM fact_transactions LIMIT 5").df())

con.close()

=== USERS TABLE ===


,user_id,full_name,email,ssn_hashed,credit_score,created_at
0,a60da676-8ed8-493e-9419-3fc622469265,Allison Hill,donaldgarcia@example.net,b23f41afa5f47b191d275cc24f063272e9a95f5b36225f...,847,2024-10-18
1,6bfe5f11-4d06-4998-a9ba-99dd0af50c58,Allen Robinson,hoffmanjennifer@example.net,aa91bb1289552259250e1b2a3cc07a788779cc744490e7...,835,2025-01-24
2,9f726db0-c63b-4b97-8bcd-41d25a9ea1f3,Rhonda Smith,helenpeterson@example.org,b148641d5fd710c862eee130b08b9963cef6207876a8b1...,780,2026-02-25
3,46dd1af4-08c6-4b6d-9e14-30ae8f42caa0,James Howard,cassandra07@example.net,55f7e1eba55c9e0b77ab1e2a3e310a17198883525b6ba4...,691,2026-06-21
4,9ccce858-690f-49d1-ba3c-e99fc232f989,Edward Fuller,barbara10@example.net,4e132b417a4596b2edd634bb466f602660787cfd9bfa68...,828,2025-03-15



=== ACCOUNTS TABLE ===


,account_id,user_id,account_type,current_balance,status
0,4caa49d3-8948-46de-b475-6ef88b3082f8,a60da676-8ed8-493e-9419-3fc622469265,Checking,9443.18,Active
1,036d9829-5cf3-45cd-a6b9-9afefab71c90,6bfe5f11-4d06-4998-a9ba-99dd0af50c58,High-Yield Savings,4216.64,Active
2,6ba2cbd4-dbf0-440a-8e9e-c4ba059a8b91,9f726db0-c63b-4b97-8bcd-41d25a9ea1f3,Credit Line,16283.76,Active
3,c4280ddf-0722-4282-a9a6-598f4b983c7f,9f726db0-c63b-4b97-8bcd-41d25a9ea1f3,High-Yield Savings,14059.24,Active
4,b371535d-f41f-4377-97dc-f27bdd1d7271,46dd1af4-08c6-4b6d-9e14-30ae8f42caa0,Credit Line,22065.61,Active



=== TRANSACTIONS TABLE ===


,transaction_id,account_id,transaction_date,amount,merchant_category,status
0,55c2538c-865a-4799-b312-deb3eff2765f,4caa49d3-8948-46de-b475-6ef88b3082f8,2026-08-14,922.28,Groceries,Settled
1,3e537648-6ce8-48a7-8f12-871659e5b393,4caa49d3-8948-46de-b475-6ef88b3082f8,2026-07-22,1243.64,P2P Transfer,Settled
2,5bbf6170-8c4d-48fa-94b4-5074fdeadbd0,4caa49d3-8948-46de-b475-6ef88b3082f8,2026-07-31,1195.27,P2P Transfer,Settled
3,92ee525b-a97e-4932-b242-9bb663e645d2,4caa49d3-8948-46de-b475-6ef88b3082f8,2026-07-25,-229.35,P2P Transfer,Settled
4,efeb332c-3d30-4574-8ddd-0cb1c016857b,4caa49d3-8948-46de-b475-6ef88b3082f8,2026-07-25,1012.79,Online Retail,Settled


In [20]:
con = duckdb.connect('titan_corevault.duckdb')

query = """
    SELECT 
        u.full_name,
        u.credit_score,
        a.account_type,
        a.current_balance,
        t.transaction_date,
        t.amount AS tx_amount,
        t.merchant_category,
        s.sofr_rate
    FROM dim_users u
    JOIN dim_accounts a ON u.user_id = a.user_id
    JOIN fact_transactions t ON a.account_id = t.account_id
    CROSS JOIN (
        SELECT sofr_rate 
        FROM fact_sofr_rates 
        ORDER BY observation_date DESC 
        LIMIT 1
    ) s
    ORDER BY t.transaction_date DESC
    LIMIT 10
"""

con.execute(query).df()

,full_name,credit_score,account_type,current_balance,transaction_date,tx_amount,merchant_category,sofr_rate
0,James Howard,691,Credit Line,22065.61,2026-08-16,78.85,Groceries,3.62
1,Latoya Robbins,693,High-Yield Savings,2492.14,2026-08-16,422.97,Utilities,3.62
2,Latoya Robbins,693,High-Yield Savings,2492.14,2026-08-16,1419.22,Utilities,3.62
3,Jeff Owens,789,High-Yield Savings,15625.61,2026-08-16,1376.48,Utilities,3.62
4,Ashley Garrett,704,Checking,11189.68,2026-08-16,929.35,Utilities,3.62
5,Robert Chase,672,Credit Line,2691.01,2026-08-16,571.82,P2P Transfer,3.62
6,James Howard,691,Credit Line,22065.61,2026-08-16,128.99,Groceries,3.62
7,Carol Tucker,748,Credit Line,22964.73,2026-08-16,1006.78,Groceries,3.62
8,Melinda Jones,827,Checking,14607.57,2026-08-16,834.03,Groceries,3.62
9,Andrew Ruiz,697,Checking,7863.52,2026-08-16,1110.26,Groceries,3.62


In [19]:
import duckdb

con = duckdb.connect('titan_corevault.duckdb')

print("Total Users:", con.execute("SELECT COUNT(*) FROM dim_users").fetchone()[0])
print("Total Accounts:", con.execute("SELECT COUNT(*) FROM dim_accounts").fetchone()[0])
print("Total Transactions:", con.execute("SELECT COUNT(*) FROM fact_transactions").fetchone()[0])
print("Total SOFR Rate Days:", con.execute("SELECT COUNT(*) FROM fact_sofr_rates").fetchone()[0])

con.close()

Total Users: 100
Total Accounts: 148
Total Transactions: 1097
Total SOFR Rate Days: 2090


In [26]:
import duckdb

con = duckdb.connect("titan_corevault.duckdb")

print("Building Phase 1 Financial Logic Views inside DuckDB...")

# VIEW 1: Daily Interest Yield Engine (High-Yield Savings)

con.execute("""
CREATE OR REPLACE VIEW view_daily_savings_yield AS
SELECT
    u.full_name,
    u.email,
    a.account_id,
    a.current_balance,
    s.observation_date AS rate_date,
    s.sofr_rate,
    (s.sofr_rate + 0.50) AS gross_apy,
    -- Daily payout formula: Balance * (APY / 100) / 365
    ROUND(a.current_balance * ((s.sofr_rate + 0.50) / 100 / 365), 4) AS daily_interest_earned,
    -- Monthly projected payout
    ROUND(a.current_balance * ((s.sofr_rate + 0.50) / 100 / 12), 2) AS projected_monthly_yield
    FROM dim_users u
    JOIN dim_accounts a ON u.user_id = a.user_id
CROSS JOIN (
    SELECT observation_date, sofr_rate 
    FROM fact_sofr_rates 
    ORDER BY observation_date DESC 
    LIMIT 1
) s
WHERE a.account_type = 'High-Yield Savings' AND a.status = 'Active';
""")

# VIEW 2: Dynamic Risk-Based Credit Line Pricing

con.execute("""
CREATE OR REPLACE VIEW view_credit_risk_pricing AS
SELECT 
    u.full_name,
    u.credit_score,
    CASE
        WHEN u.credit_score >= 750 THEN 'Prime (Tier 1)'
        WHEN u.credit_score >= 670 THEN 'Near-Prime (Tier 2)'
        ELSE 'Subprime (Tier 3)'
    END AS credit_tier,
    a.account_id,
    a.current_balance AS line_used,
    s.sofr_rate AS base_sofr_rate,
    -- Pricing Rules
    CASE
        WHEN u.credit_score >= 750 THEN 4.00
        WHEN u.credit_score >= 670 THEN 7.50
        ELSE 12.00
    END as risk_spread,
    -- Dynamic Variable APR = Base SOFR + Risk Spread
    CASE
        WHEN u.credit_score >= 750 THEN s.sofr_rate + 4.00
        WHEN u.credit_score >= 670 THEN s.sofr_rate + 7.50
        ELSE s.sofr_rate + 12.00
    END AS assigned_variable_apr
FROM dim_users u
JOIN dim_accounts a ON u.user_id = a.user_id
CROSS JOIN (
    SELECT sofr_rate 
    FROM fact_sofr_rates 
    ORDER BY observation_date DESC 
    LIMIT 1
) s
WHERE a.account_type = 'Credit Line';
""")

# VIEW 3: Customer LTV & Portfolio Health

con.execute("""
CREATE OR REPLACE VIEW view_customer_portfolio_summary AS
SELECT 
    u.user_id,
    u.full_name,
    u.credit_score,
    COUNT(DISTINCT a.account_id) AS total_accounts,
    ROUND(SUM(a.current_balance), 2) AS total_user_deposits,
    COUNT(t.transaction_id) AS total_30d_transactions,
    ROUND(SUM(CASE WHEN t.amount > 0 THEN t.amount ELSE 0 END), 2) AS total_inflow_30d,
    ROUND(SUM(CASE WHEN t.amount < 0 THEN ABS(t.amount) ELSE 0 END), 2) AS total_outflow_30d
FROM dim_users u
LEFT JOIN dim_accounts a ON u.user_id = a.user_id
LEFT JOIN fact_transactions t ON a.account_id = t.account_id
GROUP BY u.user_id, u.full_name, u.credit_score;
""")

print("Successfully created all 3 Phase 1 views in DuckDB!")

Building Phase 1 Financial Logic Views inside DuckDB...
Successfully created all 3 Phase 1 views in DuckDB!


In [36]:
%%sql
SELECT full_name, current_balance, sofr_rate, gross_apy, daily_interest_earned, projected_monthly_yield
FROM view_daily_savings_yield
ORDER BY current_balance DESC
LIMIT 5;

,full_name,current_balance,sofr_rate,gross_apy,daily_interest_earned,projected_monthly_yield
0,Richard Gibson,24939.00,3.62,4.12,2.8150,85.62
1,Joel Williams,24707.80,3.62,4.12,2.7889,84.83
2,Kyle Francis,24408.64,3.62,4.12,2.7552,83.80
3,Katherine Miller,24344.55,3.62,4.12,2.7479,83.58
4,David Douglas Jr.,24343.03,3.62,4.12,2.7478,83.58


In [5]:
%%sql
SELECT full_name, credit_score, credit_tier, line_used, base_sofr_rate, risk_spread, assigned_variable_apr
FROM view_credit_risk_pricing
ORDER BY credit_score ASC
LIMIT 5;

UsageError: Cell magic `%%sql` not found.


In [41]:
%%sql
SELECT full_name, credit_score, total_accounts, total_user_deposits, total_30d_transactions, total_inflow_30d
FROM view_customer_portfolio_summary
ORDER BY total_user_deposits DESC
LIMIT 10;

,full_name,credit_score,total_accounts,total_user_deposits,total_30d_transactions,total_inflow_30d
0,Kathleen Moran,753,2,402956.26,18,9145.20
1,Kyle Francis,641,2,379081.48,17,9394.13
2,Richard Gibson,727,2,340041.30,16,8160.22
3,Marc Hart,650,2,335937.14,19,10987.60
4,Carol Tucker,748,2,335699.46,16,9362.59
5,Melinda Jones,827,2,318068.50,18,6978.10
6,Joseph Ramos,734,2,309602.37,16,9218.38
7,Sara Calhoun,655,2,276326.92,17,10640.16
8,Robert Chase,672,2,266716.90,20,15140.72
9,Juan Moore,836,2,263231.19,15,6126.04


In [43]:
import duckdb
import pandas as pd
import requests

con = duckdb.connect("titan_corevault.duckdb")

target_companies = {
    'Apple.Inc': {'cik': '0000320193', 'ticker': 'AAPL'},
    'Microsoft Corp': {'cik': '0000789019', 'ticker': 'MSFT'},
    'JPMorgan Chase': {'cik': '0000019617', 'ticker': 'JPM'},
}

headers = {'User-Agent': 'TitanCoreVault Engine nathaniel.pfeifer@wsu.edu'}

all_financial_records = []

print('Standing SEC EDGAR API Ingestion Engine...')

for company_name, info in target_companies.items():
    cik = info['cik']
    ticker = info['ticker']
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'

    print(f'Fetching SEC XBRL facts for {company_name} ({ticker})...')
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        us_gaap = data.get('facts', {}).get('us-gaap', {})

        metrics = {
            'Assets': 'Total Assets',
            'Liabilities': 'Total Liabilities',
            'StockholdersEquity': 'Stockholders Equity',
            'OperatingIncomeLoss': 'Operating Income',
        }

        for gaap_tag, metric_label in metrics.items():
            if gaap_tag in us_gaap:
                units = us_gaap[gaap_tag].get('units', {})
                currency = (
                    'USD'
                    if 'USD' in units
                    else list(units.keys())[0]
                    if units
                    else None
                )

                if currency:
                    units_data = units[currency]
                    for item in units_data:
                        if item.get('form') == '10-K' and 'frame' in item:
                            all_financial_records.append({
                                'ticker': ticker,
                                'company_name': company_name,
                                'cik': cik,
                                'fiscal_year': item.get('fy'),
                                'metric_name': metric_label,
                                'val': item.get('val'),
                                'filed_date': item.get('filed'),
                            })
    else:
        print(
            f'Failed to fetch {company_name}: Status Code {response.status_code}'
        )

if all_financial_records:
    df_sec = pd.DataFrame(all_financial_records)
    df_sec = df_sec.drop_duplicates(
        subset=['ticker', 'fiscal_year', 'metric_name']
    )
    print(f'\nExtracted {len(df_sec)} corporate filing metrics.')

    df_pivoted = df_sec.pivot(
        index=['ticker', 'company_name', 'cik', 'fiscal_year'],
        columns='metric_name',
        values='val',
    ).reset_index()

    df_pivoted.columns.name = None

    con.execute('DROP TABLE IF EXISTS fact_sec_financials;')
    con.execute('CREATE TABLE fact_sec_financials AS SELECT * FROM df_pivoted;')

    print(
        "Successfully loaded SEC EDGAR financial metrics into DuckDB ('fact_sec_financials')!"
    )
else:
    print('No financial records retrieved. Check SEC headers or API responses.')

Standing SEC EDGAR API Ingestion Engine...
Fetching SEC XBRL facts for Apple.Inc (AAPL)...
Fetching SEC XBRL facts for Microsoft Corp (MSFT)...
Fetching SEC XBRL facts for JPMorgan Chase (JPM)...

Extracted 158 corporate filing metrics.
Successfully loaded SEC EDGAR financial metrics into DuckDB ('fact_sec_financials')!


In [44]:
%%sql
SELECT *
FROM fact_sec_financials
LIMIT 10;

,ticker,company_name,cik,fiscal_year,Operating Income,Stockholders Equity,Total Assets,Total Liabilities
0,AAPL,Apple.Inc,0000320193,2010,8.327000e+09,1.453100e+10,3.617100e+10,1.586100e+10
1,AAPL,Apple.Inc,0000320193,2011,1.174000e+10,2.229700e+10,4.750100e+10,2.739200e+10
2,AAPL,Apple.Inc,0000320193,2013,3.379000e+10,4.779100e+10,1.760640e+11,5.785400e+10
3,AAPL,Apple.Inc,0000320193,2015,4.899900e+10,1.182100e+11,2.318390e+11,1.202920e+11
4,AAPL,Apple.Inc,0000320193,2016,5.250300e+10,1.235490e+11,2.903450e+11,1.709900e+11
5,AAPL,Apple.Inc,0000320193,2017,7.123000e+10,1.115470e+11,3.216860e+11,1.934370e+11
6,AAPL,Apple.Inc,0000320193,2018,6.002400e+10,1.193550e+11,3.753190e+11,2.412720e+11
7,AAPL,Apple.Inc,0000320193,2019,6.134400e+10,1.282490e+11,3.657250e+11,2.585780e+11
8,AAPL,Apple.Inc,0000320193,2020,7.089800e+10,1.340470e+11,3.385160e+11,2.480280e+11
9,AAPL,Apple.Inc,0000320193,2021,6.393000e+10,1.071470e+11,3.238880e+11,2.585490e+11


In [45]:
%%sql
CREATE OR REPLACE VIEW view_corporate_credit_risk AS
SELECT
    f.company_name,
    f.ticker,
    f.fiscal_year,
    f."Total Assets" AS total_assets,
    f."Total Liabilities" AS total_liabilities,
    f."Stockholders Equity" AS equity,
    ROUND(f."Total Liabilities" / NULLIF(f."Stockholders Equity", 0), 2) AS debt_to_equity_ratio
FROM fact_sec_financials f
WHERE f.fiscal_year IS NOT NULL;

,Count


In [48]:
%%sql
SElECT *
FROM view_corporate_credit_risk
ORDER BY ticker, fiscal_year DESC
LIMIT 15;

,company_name,ticker,fiscal_year,total_assets,total_liabilities,equity,debt_to_equity_ratio
0,Apple.Inc,AAPL,2025,3.649800e+11,3.080300e+11,5.067200e+10,6.08
1,Apple.Inc,AAPL,2024,3.525830e+11,2.904370e+11,6.309000e+10,4.60
2,Apple.Inc,AAPL,2023,3.527550e+11,3.020830e+11,6.533900e+10,4.62
3,Apple.Inc,AAPL,2022,3.510020e+11,2.879120e+11,9.048800e+10,3.18
4,Apple.Inc,AAPL,2021,3.238880e+11,2.585490e+11,1.071470e+11,2.41
5,Apple.Inc,AAPL,2020,3.385160e+11,2.480280e+11,1.340470e+11,1.85
6,Apple.Inc,AAPL,2019,3.657250e+11,2.585780e+11,1.282490e+11,2.02
7,Apple.Inc,AAPL,2018,3.753190e+11,2.412720e+11,1.193550e+11,2.02
8,Apple.Inc,AAPL,2017,3.216860e+11,1.934370e+11,1.115470e+11,1.73
9,Apple.Inc,AAPL,2016,2.903450e+11,1.709900e+11,1.235490e+11,1.38


In [49]:
%%sql
WITH financial_metrics AS (
    SELECT
        company_name,
        ticker,
        fiscal_year,
        total_assets,
        total_liabilities,
        debt_to_equity_ratio,
        LAG(total_assets) OVER (
            PARTITION BY ticker
            ORDER BY fiscal_year ASC
        ) AS prev_year_assets,
        LAG(debt_to_equity_ratio) OVER (
            PARTITION BY ticker
            ORDER BY fiscal_year ASC
        ) AS prev_year_de_ratio
    FROM view_corporate_credit_risk
)
SELECT
    ticker,
    fiscal_year,
    ROUND(total_assets / 1e9, 2) AS assets_billions,
    debt_to_equity_ratio,
    ROUND(
        ((total_assets - prev_year_assets) / NULLIF(prev_year_assets, 0)) * 100,
        2
    ) AS yoy_asset_growth_pct,
    
    ROUND(debt_to_equity_ratio - prev_year_de_ratio, 2) AS de_ratio_change,

    CASE
        WHEN ticker = 'JPM' AND debt_to_equity_ratio > 10.0 THEN 'Bank Standard (Deposit Heavy)'
        WHEN debt_to_equity_ratio > 3.0 THEN 'High Leverage Risk'
        WHEN debt_to_equity_ratio BETWEEN 1.5 AND 3.0 THEN 'Moderate Leverage'
        ELSE 'Conservative Capital Structure'
    END AS risk_profile
FROM financial_metrics
ORDER BY ticker ASC, fiscal_year DESC;

,ticker,fiscal_year,assets_billions,debt_to_equity_ratio,yoy_asset_growth_pct,de_ratio_change,risk_profile
0,AAPL,2025,364.98,6.08,3.52,1.48,High Leverage Risk
1,AAPL,2024,352.58,4.60,-0.05,-0.02,High Leverage Risk
2,AAPL,2023,352.76,4.62,0.50,1.44,High Leverage Risk
3,AAPL,2022,351.00,3.18,8.37,0.77,High Leverage Risk
4,AAPL,2021,323.89,2.41,-4.32,0.56,Moderate Leverage
5,AAPL,2020,338.52,1.85,-7.44,-0.17,Moderate Leverage
6,AAPL,2019,365.73,2.02,-2.56,0.00,Moderate Leverage
7,AAPL,2018,375.32,2.02,16.67,0.29,Moderate Leverage
8,AAPL,2017,321.69,1.73,10.79,0.35,Moderate Leverage
9,AAPL,2016,290.35,1.38,25.24,0.36,Conservative Capital Structure


In [52]:
import duckdb
import plotly.express as px

con = duckdb.connect('titan_corevault.duckdb')

df_risk = con.execute('''
    SELECT
        company_name,
        ticker,
        fiscal_year,
        debt_to_equity_ratio
    FROM view_corporate_credit_risk
    WHERE fiscal_year >= 2015
    ORDER BY ticker, fiscal_year ASC
''').df()

fig = px.line(
    df_risk,
    x='fiscal_year',
    y='debt_to_equity_ratio',
    color='ticker',
    symbol='ticker',
    title='TitanCoreVault: Corporate Capital Structure & Leverage Trends (2015–2025)',
    labels={
        'fiscal_year': 'Fiscal Year',
        'debt_to_equity_ratio': 'Debt-to-Equity-Ratio',
        'ticker': 'Ticker',
    },
hover_data=['company_name'],
template='plotly_white',
)
    
fig.update_traces(marker=dict(size=8))
fig.update_layout(
    xaxis=dict(dtick=1),
    yaxis_title='Debt-to-Equity Ratio (x)',
    legend_title_text="Company",
    font=dict(family='Arial, sans-serif', size=12) 
)
fig.show()

Writing app.py


In [55]:
import duckdb
duckdb.close()

In [23]:
import duckdb
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

con = duckdb.connect('titan_corevault.duckdb')
df_export = con.execute('''
    SELECT
        company_name AS 'Company Name',
        ticker AS 'Ticker',
        fiscal_year AS 'Fiscal Year',
        total_assets AS 'Total Assets ($)',
        total_liabilities AS 'Total Liabilities ($)',
        equity AS 'Stockholders Equity ($)',
        debt_to_equity_ratio AS 'Debt_to_Equity Ratio'
    FROM view_corporate_credit_risk
    ORDER BY ticker, fiscal_year DESC
''').df()
con.close()

excel_filename = 'corporate_credit_risk_report.xlsx'
df_export.to_excel(excel_filename, index=False, sheet_name='Credit Risk Ledger')

wb = openpyxl.load_workbook(excel_filename)
ws = wb["Credit Risk Ledger"]

header_font = Font(name="Arial", size=11, bold=True, color="FFFFFF")
header_fill = PatternFill(
    start_color="1F4E78", end_color="1F4E78", fill_type="solid"
)
data_font = Font(name="Arial", size=10)
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

for col_num in range(1, ws.max_column + 1):
    cell = ws.cell(row=1, column=col_num)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(
        horizontal="center", vertical="center", wrap_text=True
    )

for row in ws.iter_rows(min_row=2, max_row=ws.max_row, max_col=ws.max_column):
    for cell in row:
        cell.font = data_font
        cell.border = thin_border

        col_letter = get_column_letter(cell.column)

        if cell.column in [4,5,6]:
            cell.number_format = '$#,##0'
            cell.alignment = Alignment(horizontal='right')
        elif cell.column == 7:
            cell.number_format = '0.00x'
            cell.alignment = Alignment(horizontal='right')
        else:
            cell.alignment = Alignment(horizontal='center')

for col in ws.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

wb.save(excel_filename)
print(
    f'Successfully generated formatted report: {excel_filename}'
)



Successfully generated formatted report: corporate_credit_risk_report.xlsx


In [20]:
import duckdb

con = duckdb.connect("titan_corevault.duckdb")

con.execute('''
UPDATE fact_sec_financials
SET
    "Total Assets" = 2351698000000,
    "Total Liabilities" = 2104125000000,
    "Stockholders Equity" = 247573000000
WHERE ticker = 'JPM' AND fiscal_year = 2015;
''')

con.execute("""
CREATE OR REPLACE VIEW view_corporate_credit_risk AS
SELECT 
    f.company_name,
    f.ticker,
    f.fiscal_year,
    f."Total Assets" AS total_assets,
    f."Total Liabilities" AS total_liabilities,
    f."Stockholders Equity" AS equity,
    ROUND(f."Total Liabilities" / NULLIF(f."Stockholders Equity", 0), 2) AS debt_to_equity_ratio
FROM fact_sec_financials f;
""")

con.close()
print("Updated database table successfully!")

Updated database table successfully!


In [21]:
import duckdb

con = duckdb.connect("titan_corevault.duckdb")
df_check = con.execute("""
    SELECT ticker, fiscal_year, total_assets, total_liabilities, equity, debt_to_equity_ratio 
    FROM view_corporate_credit_risk 
    WHERE ticker = 'JPM' AND fiscal_year BETWEEN 2013 AND 2017
    ORDER BY fiscal_year;
""").df()
con.close()

print(df_check)

  ticker  fiscal_year  total_assets  total_liabilities        equity  \
0    JPM         2013  2.265792e+12       2.155072e+12  2.040690e+11   
1    JPM         2014  2.359141e+12       2.204511e+12  2.111780e+11   
2    JPM         2015  2.351698e+12       2.104125e+12  2.475730e+11   
3    JPM         2016  2.572274e+12       2.104125e+12  2.317270e+11   
4    JPM         2017  2.351698e+12       2.236782e+12  2.475730e+11   

   debt_to_equity_ratio  
0                 10.56  
1                 10.44  
2                  8.50  
3                  9.08  
4                  9.03  


In [12]:
import duckdb

con = duckdb.connect("titan_corevault.duckdb")
print(con.execute("SHOW TABLES").df())
con.close()

                              name
0                     dim_accounts
1                        dim_users
2              fact_sec_financials
3                  fact_sofr_rates
4                fact_transactions
5       view_corporate_credit_risk
6         view_credit_risk_pricing
7  view_customer_portfolio_summary
8         view_daily_savings_yield


In [22]:
import duckdb

con = duckdb.connect("titan_corevault.duckdb")
df_check = con.execute("""
    SELECT ticker, fiscal_year, 
           ROUND(total_assets / 1e9, 2) AS assets_B,
           ROUND(total_liabilities / 1e9, 2) AS liab_B,
           ROUND(equity / 1e9, 2) AS equity_B,
           debt_to_equity_ratio
    FROM view_corporate_credit_risk
    WHERE ticker = 'JPM' AND fiscal_year BETWEEN 2013 AND 2017
    ORDER BY fiscal_year ASC
""").df()

con.close()
print(df_check)

  ticker  fiscal_year  assets_B   liab_B  equity_B  debt_to_equity_ratio
0    JPM         2013   2265.79  2155.07    204.07                 10.56
1    JPM         2014   2359.14  2204.51    211.18                 10.44
2    JPM         2015   2351.70  2104.13    247.57                  8.50
3    JPM         2016   2572.27  2104.13    231.73                  9.08
4    JPM         2017   2351.70  2236.78    247.57                  9.03


In [25]:
try:
    df_data = load_credit_risk_data()
except Exception as e:
    st.error(f"Error loading database: {e}")
    st.stop()

2026-08-19 17:09:27.086 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-19 17:09:27.088 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-19 17:09:27.089 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-19 17:09:27.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-19 17:09:27.179 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-19 17:09:27.179 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-19 17:09:27.180 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-19 17:09:27.180 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [34]:
import random
import uuid
import duckdb
from faker import Faker

fake = Faker("en_US")
Faker.seed(101)

con = duckdb.connect("titan_corevault.duckdb")

new_users = []
for _ in range(50):
    new_users.append(
        (
            str(uuid.uuid4()),
            fake.name(),
            fake.email(),
            fake.sha256(),
            random.randint(580, 850),
            fake.date_between(start_date="-2y", end_date="today"),
        )
    )

con.executemany(
    "INSERT INTO dim_users VALUES (?, ?, ?, ?, ?, ?)", new_users
)

new_accounts = []
new_transactions = []
account_types = ["High-Yield Savings", "Checking", "Credit Line"]

for user in new_users:
    user_id = user[0]
    for _ in range(random.choice([1, 2])):
        acct_id = str(uuid.uuid4())
        acct_type = random.choice(account_types)

        new_accounts.append(
            (
                acct_id,
                user_id,
                acct_type,
                round(random.uniform(1000.0, 35000.0), 2),
                "Active",
            )
        )

        for _ in range(random.randint(10, 25)):
            new_transactions.append(
                (
                    str(uuid.uuid4()),
                    acct_id,
                    fake.date_between(start_date="-1y", end_date="today"),
                    round(random.uniform(-500.0, 1500.0), 2),
                    random.choice(
                        [
                            "Groceries",
                            "Payroll Direct Deposit",
                            "P2P Transfer",
                            "Utilities",
                            "Online Retail",
                        ]
                    ),
                    "Settled",
                )
            )

con.executemany(
    "INSERT INTO dim_accounts VALUES (?, ?, ?, ?, ?)", new_accounts
)
con.executemany(
    "INSERT INTO fact_transactions VALUES (?, ?, ?, ?, ?, ?)", new_transactions
)

con.close()
print("Non-corporate data backfill complete.")

Non-corporate data backfill complete.


In [35]:
import duckdb

con = duckdb.connect("titan_corevault.duckdb")

print("=== CORE VAULT DATA AUDIT ===")
audit_df = con.execute("""
SELECT 
    COUNT(DISTINCT u.user_id) AS total_users,
    COUNT(DISTINCT a.account_id) AS total_accounts,
    COUNT(t.transaction_id) AS total_transactions,
    MIN(t.transaction_date) AS earliest_transaction,
    MAX(t.transaction_date) AS latest_transaction,
    ROUND(SUM(CASE WHEN t.amount > 0 THEN t.amount ELSE 0 END), 2) AS total_inflow,
    ROUND(SUM(CASE WHEN t.amount < 0 THEN ABS(t.amount) ELSE 0 END), 2) AS total_outflow
FROM dim_users u
JOIN dim_accounts a ON u.user_id = a.user_id
JOIN fact_transactions t ON a.account_id = t.account_id;
""").df()

display(audit_df)

print("\n=== MONTHLY TRANSACTION DENSITY ===")
timeline_df = con.execute("""
SELECT 
    DATE_TRUNC('month', transaction_date) AS month,
    COUNT(transaction_id) AS monthly_transactions,
    ROUND(SUM(amount), 2) AS net_cash_flow
FROM fact_transactions
GROUP BY 1
ORDER BY 1 ASC;
""").df()

display(timeline_df)

con.close()

=== CORE VAULT DATA AUDIT ===


,total_users,total_accounts,total_transactions,earliest_transaction,latest_transaction,total_inflow,total_outflow
0,150,225,2526,2025-08-19,2026-08-19,1431003.41,152325.06



=== MONTHLY TRANSACTION DENSITY ===


,month,monthly_transactions,net_cash_flow
0,2025-08-01,44,23994.62
1,2025-09-01,127,61687.96
2,2025-10-01,110,52300.68
3,2025-11-01,116,55215.44
4,2025-12-01,132,73902.30
5,2026-01-01,129,61378.20
6,2026-02-01,100,42800.97
7,2026-03-01,120,63784.55
8,2026-04-01,116,55098.73
9,2026-05-01,125,59019.63


In [2]:
%%writefile app.py

import io
import duckdb
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

st.set_page_config(
    page_title="Titan Pay CoreVault Risk Engine",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown(
    """
    <style>
    .main-header { font-size: 2.2rem; font-weight: 700; color: #1F4E78; margin-bottom: 0px; }
    .sub-header { font-size: 1.0rem; color: #555555; margin-bottom: 20px; }
    
    .context-box {
        background-color: var(--background-secondary-color, rgba(128, 128, 128, 0.1));
        color: var(--text-color, inherit);
        border-left: 5px solid #1F4E78;
        padding: 12px 18px;
        margin-bottom: 10px;
        border-radius: 4px;
    }
    
    .metric-badge-green {
        background-color: #D4EDDA; color: #155724; padding: 6px 12px; 
        border-radius: 6px; font-weight: bold; display: inline-block;
    }
    .metric-badge-yellow {
        background-color: #FFF3CD; color: #856404; padding: 6px 12px; 
        border-radius: 6px; font-weight: bold; display: inline-block;
    }
    .metric-badge-red {
        background-color: #F8D7DA; color: #721C24; padding: 6px 12px; 
        border-radius: 6px; font-weight: bold; display: inline-block;
    }
    </style>
""",
    unsafe_allow_html=True,
)

DB_PATH = "titan_corevault.duckdb"

@st.cache_data(ttl=600)
def load_credit_risk_data():
    con = duckdb.connect(DB_PATH, read_only=False)
    
    con.execute('''
    CREATE OR REPLACE VIEW view_comprehensive_credit_risk AS
    SELECT
        f.company_name AS account_name,
        f.ticker AS account_id,
        'Corporate' AS account_category,
        f.fiscal_year,
        f."Total Assets" AS total_assets,
        f."Total Liabilities" AS total_liabilities,
        f."Stockholders Equity" AS equity,
        f."Operating Income" AS operating_income,
        ROUND(f."Total Liabilities" / NULLIF(f."Stockholders Equity", 0), 2) AS debt_to_equity,
        ROUND(f."Operating Income" / NULLIF(f."Total Liabilities" * 0.035, 0), 2) AS interest_coverage_ratio,
        CASE 
            WHEN f.ticker = 'JPM' THEN NULL
            ELSE ROUND(
                (6.56 * ((f."Total Assets" - f."Total Liabilities") / NULLIF(f."Total Assets", 0))) +
                (3.26 * (f."Stockholders Equity" / NULLIF(f."Total Assets", 0))) +
                (6.72 * (f."Operating Income" / NULLIF(f."Total Assets", 0))) +
                (1.05 * (f."Stockholders Equity" / NULLIF(f."Total Liabilities", 0))), 2
            )
        END AS altman_z_score,
        CASE WHEN f.ticker = 'JPM' THEN TRUE ELSE FALSE END AS is_financial_institution
    FROM fact_sec_financials f

    UNION ALL

    SELECT 'Tier 1: Prime / Low Risk', 'TIER_1_PRIME', 'Credit Tier', 2024, 5000000000, 1500000000, 3500000000, 450000000, 0.43, 8.50, NULL, FALSE
    UNION ALL
    SELECT 'Tier 1: Prime / Low Risk', 'TIER_1_PRIME', 'Credit Tier', 2025, 5500000000, 1600000000, 3900000000, 500000000, 0.41, 8.90, NULL, FALSE
    
    UNION ALL
    
    SELECT 'Tier 2: Mid-Grade / Moderate', 'TIER_2_MID', 'Credit Tier', 2024, 2500000000, 1800000000, 700000000, 150000000, 2.57, 2.38, NULL, FALSE
    UNION ALL
    SELECT 'Tier 2: Mid-Grade / Moderate', 'TIER_2_MID', 'Credit Tier', 2025, 2700000000, 1950000000, 750000000, 165000000, 2.60, 2.42, NULL, FALSE

    UNION ALL

    SELECT 'Tier 3: Subprime / High Risk', 'TIER_3_SUBPRIME', 'Credit Tier', 2024, 1000000000, 920000000, 80000000, 35000000, 11.50, 1.08, NULL, FALSE
    UNION ALL
    SELECT 'Tier 3: Subprime / High Risk', 'TIER_3_SUBPRIME', 'Credit Tier', 2025, 1050000000, 980000000, 70000000, 30000000, 14.00, 0.87, NULL, FALSE;
    ''')

    df = con.execute(
        "SELECT * FROM view_comprehensive_credit_risk ORDER BY account_category, account_id, fiscal_year DESC"
    ).df()
    con.close()
    return df

@st.cache_data(ttl=600)
def load_monthly_transaction_data():
    con = duckdb.connect(DB_PATH, read_only=True)
    df_tx = con.execute('''
        SELECT 
            DATE_TRUNC('month', transaction_date) AS month,
            a.account_type,
            COUNT(t.transaction_id) AS transaction_count,
            ROUND(SUM(t.amount), 2) AS net_cash_flow,
            ROUND(SUM(CASE WHEN t.amount > 0 THEN t.amount ELSE 0 END), 2) AS total_inflow,
            ROUND(SUM(CASE WHEN t.amount < 0 THEN ABS(t.amount) ELSE 0 END), 2) AS total_outflow
        FROM fact_transactions t
        JOIN dim_accounts a ON t.account_id = a.account_id
        GROUP BY 1, 2
        ORDER BY month ASC
    ''').df()
    con.close()
    return df_tx

@st.cache_data(ttl=600)
def load_monthly_tier_data():
    months = pd.date_range(start="2024-01-01", end="2025-12-01", freq="MS")
    records = []
    
    for m in months:
        progress = (m.month - 1) / 12.0
        if m.year == 2024:
            t1_de, t2_de, t3_de = 0.43 + (0.02 * progress), 2.57 + (0.03 * progress), 11.50 + (1.25 * progress)
        else:
            t1_de, t2_de, t3_de = 0.41 - (0.01 * progress), 2.60 - (0.02 * progress), 12.75 + (1.25 * progress)
            
        records.append({"month": m, "account_id": "TIER_1_PRIME", "debt_to_equity": round(t1_de, 2)})
        records.append({"month": m, "account_id": "TIER_2_MID", "debt_to_equity": round(t2_de, 2)})
        records.append({"month": m, "account_id": "TIER_3_SUBPRIME", "debt_to_equity": round(t3_de, 2)})
        
    return pd.DataFrame(records)

def generate_excel_report(df):
    output = io.BytesIO()
    df_export = df.copy()
    df_export.columns = [
        "Account / Tier Name", "Account ID / Code", "Category", "Fiscal Year",
        "Total Assets ($)", "Total Liabilities ($)", "Equity / Cushion ($)",
        "Operating / Annual Income ($)", "Leverage Ratio (D/E)", "Interest Coverage Ratio",
        "Altman Z-Score", "Is Bank / FI"
    ]

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Credit Risk & Tier Ledger"
    ws.append(list(df_export.columns))

    for row in df_export.itertuples(index=False):
        ws.append(list(row))

    header_font = Font(name="Arial", size=11, bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    data_font = Font(name="Arial", size=10)
    thin_border = Border(
        left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"),
        top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9")
    )

    for col_num in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, max_col=ws.max_column):
        for cell in row:
            cell.font = data_font
            cell.border = thin_border
            if cell.column in [5, 6, 7, 8]:
                cell.number_format = "$#,##0"
                cell.alignment = Alignment(horizontal="right")
            elif cell.column in [9, 10]:
                cell.number_format = "0.00x"
                cell.alignment = Alignment(horizontal="right")
            elif cell.column == 11:
                cell.number_format = "0.00" if pd.notna(cell.value) else "@"
                cell.alignment = Alignment(horizontal="right")
            else:
                cell.alignment = Alignment(horizontal="center")

    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

    wb.save(output)
    output.seek(0)
    return output

try:
    df_data = load_credit_risk_data()
    df_transactions = load_monthly_transaction_data()
    df_monthly_tiers = load_monthly_tier_data()
except Exception as e:
    st.error(f"Error loading database: {e}")
    st.stop()

st.sidebar.image("https://raw.githubusercontent.com/twitter/twemoji/master/assets/svg/1f6e1.svg", width=65)
st.sidebar.title("Titan Pay CoreVault")
st.sidebar.markdown("**Unified Credit & Risk Engine**")
st.sidebar.markdown("---")

category_filter = st.sidebar.radio(
    "Select Segment View:",
    options=["ALL", "Corporate Accounts", "Non-Corporate Risk Tiers"]
)

filtered_by_cat = df_data.copy()
if category_filter == "Corporate Accounts":
    filtered_by_cat = filtered_by_cat[filtered_by_cat["account_category"] == "Corporate"]
elif category_filter == "Non-Corporate Risk Tiers":
    filtered_by_cat = filtered_by_cat[filtered_by_cat["account_category"] == "Credit Tier"]

selected_account = st.sidebar.selectbox(
    "Select Entity / Risk Tier:",
    options=["ALL"] + list(filtered_by_cat["account_id"].unique()),
)

year_range = st.sidebar.slider(
    "Fiscal Year Range:",
    int(df_data["fiscal_year"].min()),
    int(df_data["fiscal_year"].max()),
    (int(df_data["fiscal_year"].min()), int(df_data["fiscal_year"].max())),
)

st.sidebar.markdown("---")
st.sidebar.subheader("Audit & Compliance Export")
excel_bytes = generate_excel_report(filtered_by_cat)
st.sidebar.download_button(
    label="📥 Download Audit Ledger (.xlsx)",
    data=excel_bytes,
    file_name="titan_pay_credit_risk_report.xlsx",
    mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
)

filtered_df = filtered_by_cat[
    (filtered_by_cat["fiscal_year"] >= year_range[0]) &
    (filtered_by_cat["fiscal_year"] <= year_range[1])
]

if selected_account != "ALL":
    filtered_df = filtered_df[filtered_df["account_id"] == selected_account]

st.markdown('<p class="main-header">Titan CoreVault Risk Management Dashboard</p>', unsafe_allow_html=True)

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Executive Summary", 
    "📈 Corporate Insolvency", 
    "📝 Underwriting & Valuation", 
    "💳 Portfolio Analytics", 
    "📋 Data Ledger"
])

with tab1:
    with st.expander("ℹ️ View Methodology & Context (Executive Summary)", expanded=False):
        st.markdown("""
            <div class="context-box">
                <b>📌 Situation:</b> Titan Pay needed a unified platform to monitor both multi-billion dollar corporate clients and retail credit tiers.<br>
                <b>💡 Solution:</b> CoreVault integrates SEC financial data, DuckDB analytics, and dynamic sensitivity engines into a real-time risk platform.<br>
                <b>🔍 What This Means:</b> Executive leadership gains an instant portfolio health check, dynamic exposure totals, and audit-ready data.
            </div>
        """, unsafe_allow_html=True)

    st.subheader('Executive Risk & Exposure Summary')
    kpi1, kpi2, kpi3, kpi4 = st.columns(4)

    total_liabilities = filtered_df['total_liabilities'].sum()
    latest_records = filtered_df.sort_values('fiscal_year').groupby('account_id').last()

    non_bank_records = latest_records[~latest_records['is_financial_institution']]
    avg_z_score = non_bank_records["altman_z_score"].dropna().mean()
    avg_icr = latest_records["interest_coverage_ratio"].mean()

    kpi1.metric("Total Outstanding Exposure", f"${total_liabilities/1e9:,.2f}B" if total_liabilities >= 1e9 else f"${total_liabilities/1e6:,.1f}M")
    kpi2.metric("Tracked Entities / Tiers", len(filtered_df["account_id"].unique()))
    kpi3.metric(
        "Avg Corporate Z-Score",
        f"{avg_z_score:.2f}" if pd.notna(avg_z_score) else "N/A (FI / Tiers)",
        delta="Safe Zone" if (pd.notna(avg_z_score) and avg_z_score > 2.6) else "Watchlist",
    )
    kpi4.metric(
        "Avg Interest Coverage",
        f"{avg_icr:.2f}x" if pd.notna(avg_icr) else "N/A",
        delta="Strong Buffer" if (pd.notna(avg_icr) and avg_icr > 5.0) else "Elevated Risk",
    )

    st.markdown("---")
    st.subheader("🎯 Project Scope & Architecture Overview")
    
    scope_col1, scope_col2 = st.columns(2)
    with scope_col1:
        st.markdown("""
        **Core Engine Architecture:**
        * **Database Layer:** DuckDB relational engine running optimized views over SEC financials and transaction facts.
        * **Analytics Layer:** Automated financial ratio computation (D/E, Interest Coverage, Altman Z-Score, DCF Valuation).
        * **UI/UX Layer:** Streamlit-powered dashboard featuring custom CSS styling and real-time Plotly visualizations.
        """)
    with scope_col2:
        st.markdown("""
        **System Capabilities:**
        * **Multi-Segment Monitoring:** Blends SEC corporate filings with standardized non-corporate credit risk profiles.
        * **Real-time Stress Testing:** Interactive DCF sensitivity analysis for corporate valuations and default shock models for credit tiers.
        * **Audit Compliance:** Automated generation of Excel reports styled for institutional audit specifications.
        """)

with tab2:
    with st.expander("ℹ️ View Methodology & Context (Corporate Insolvency)", expanded=False):
        st.markdown("""
            <div class="context-box">
                <b>📌 Situation:</b> Corporate counterparties present varying degrees of leverage and insolvency exposure.<br>
                <b>💡 Solution:</b> Automated tracking of Debt-to-Equity (D/E) profiles alongside non-bank Altman Z-Score models.<br>
                <b>🔍 What This Means:</b> Risk teams can proactively identify deteriorating credit quality prior to default events.
            </div>
        """, unsafe_allow_html=True)

    st.subheader("Corporate Risk & Insolvency Profile")
    corp_df = filtered_df[filtered_df["account_category"] == "Corporate"]

    col_corp1, col_corp2 = st.columns(2)

    with col_corp1:
        if not corp_df.empty:
            fig_dte_corp = px.line(
                corp_df,
                x="fiscal_year",
                y="debt_to_equity",
                color="account_id",
                markers=True,
                title="Corporate Leverage Profile (Debt-to-Equity)",
                labels={"debt_to_equity": "Leverage Ratio (D/E)", "fiscal_year": "Fiscal Year", "account_id": "Corporate Ticker"},
            )
            fig_dte_corp.update_layout(hovermode='x unified')
            st.plotly_chart(fig_dte_corp, use_container_width=True)
        else:
            st.info("No corporate accounts match the current sidebar filter selection.")

    with col_corp2:
        z_score_df = corp_df[~corp_df["is_financial_institution"]]
        if not z_score_df.empty:
            fig_z = px.bar(
                z_score_df,
                x="fiscal_year",
                y='altman_z_score',
                color="account_id",
                barmode="group",
                title="Corporate Insolvency Risk Buffer (Altman Z-Score)",
                labels={'altman_z_score': "Altman Z-Score", 'fiscal_year': "Fiscal Year", 'account_id': "Corporate Ticker"},
            )
            fig_z.add_hline(
                y=2.6, line_dash='dash', line_color='#006622', line_width=2,
                annotation_text=' Safe Threshold (2.6) ', annotation_position='top left',
                annotation_font_color='#FFFFFF', annotation_font_size=12,
                annotation_font_weight='bold', annotation_bgcolor='#006622',
                annotation_bordercolor='#006622', annotation_borderwidth=3,
            )
            st.plotly_chart(fig_z, use_container_width=True)
        else:
            st.info("Altman Z-Scores apply exclusively to non-bank Corporate entities.")

with tab3:
    with st.expander("ℹ️ View Methodology & Context (Underwriting & Valuation)", expanded=False):
        st.markdown("""
            <div class="context-box">
                <b>📌 Situation:</b> Underwriters require immediate access to credit metrics and sensitivity modeling for specific counterparties.<br>
                <b>💡 Solution:</b> In-page entity filtering paired with dynamic risk-colored indicators and live DCF sensitivity models.<br>
                <b>🔍 What This Means:</b> Accelerates credit line approvals while delivering precise, color-coded risk visibility.
            </div>
        """, unsafe_allow_html=True)

    st.subheader("Credit Underwriting Briefs & Valuation Modeling")
    
    filter_col1, filter_col2 = st.columns([1, 2])
    with filter_col1:
        tab3_cat = st.radio("Filter Category:", ["ALL", "Corporate", "Credit Tier"], horizontal=True, key="tab3_cat_filter")
    with filter_col2:
        avail_entities = df_data.copy()
        if tab3_cat != "ALL":
            avail_entities = avail_entities[avail_entities["account_category"] == tab3_cat]
        entity_options = list(avail_entities["account_id"].unique())
        tab3_selected_entity = st.selectbox("🔍 Select Entity / Risk Tier for Brief:", options=entity_options, key="tab3_entity_selector")

    st.markdown("---")

    account_df = df_data[
        (df_data["account_id"] == tab3_selected_entity) & 
        (df_data["fiscal_year"] >= year_range[0]) & 
        (df_data["fiscal_year"] <= year_range[1])
    ]

    if account_df.empty:
        st.warning("No financial data available for the selected entity in this fiscal year range.")
    else:
        latest_client = account_df.sort_values('fiscal_year', ascending=False).iloc[0]
        
        de_val = latest_client['debt_to_equity']
        if de_val <= 1.5:
            de_class, de_status = "metric-badge-green", "Low Risk"
        elif de_val <= 3.5:
            de_class, de_status = "metric-badge-yellow", "Moderate Risk"
        else:
            de_class, de_status = "metric-badge-red", "High Risk / Highly Leveraged"

        icr_val = latest_client['interest_coverage_ratio']
        if icr_val >= 5.0:
            icr_class, icr_status = "metric-badge-green", "Strong Buffer"
        elif icr_val >= 2.0:
            icr_class, icr_status = "metric-badge-yellow", "Adequate Coverage"
        else:
            icr_class, icr_status = "metric-badge-red", "Distressed Coverage"

        st.markdown(f"### Entity Evaluation: **{latest_client['account_name']}** (`{latest_client['account_category']}`)")

        rec_col1, rec_col2 = st.columns([1, 2])

        with rec_col1:
            st.markdown("**Evaluative Health Indicators:**")
            st.markdown(f"• **Leverage (D/E):** <span class='{de_class}'>{de_val:.2f}x — {de_status}</span>", unsafe_allow_html=True)
            st.markdown(f"• **Interest Coverage:** <span class='{icr_class}'>{icr_val:.2f}x — {icr_status}</span>", unsafe_allow_html=True)
            
            if latest_client['is_financial_institution']:
                st.markdown("• **Altman Z-Score:** <span class='metric-badge-green'>N/A (Regulated Bank)</span>", unsafe_allow_html=True)
            elif latest_client['account_category'] == 'Corporate':
                z_val = latest_client['altman_z_score']
                if z_val >= 2.6:
                    z_class, z_status = "metric-badge-green", "Safe Zone"
                elif z_val >= 1.8:
                    z_class, z_status = "metric-badge-yellow", "Grey Zone"
                else:
                    z_class, z_status = "metric-badge-red", "Distress Zone"
                st.markdown(f"• **Altman Z-Score:** <span class='{z_class}'>{z_val:.2f} — {z_status}</span>", unsafe_allow_html=True)
            else:
                st.markdown("• **Risk Model:** `Standardized Portfolio Tier`", unsafe_allow_html=True)

        with rec_col2:
            if latest_client['is_financial_institution']:
                st.info(
                    "**Underwriting Status: APPROVED (Regulated Bank Interbank Line)**\n\n"
                    "JPMorgan Chase operates as a Tier-1 G-SIB with Fed Basel III CET1 ratios >15.0%. "
                    "Titan Pay approves interbank treasury settlement lines subject to continuous liquidity monitoring."
                )
            elif latest_client['account_category'] == 'Corporate':
                if tab3_selected_entity == "MSFT":
                    st.success("**Underwriting Status: APPROVED (Tier-1 Unsecured Line)**\n\nTitan Pay authorizes up to **$5.0 Billion** in treasury settlement with zero collateral requirements based on exceptional solvency.")
                elif tab3_selected_entity == "AAPL":
                    st.warning("**Underwriting Status: APPROVED WITH MONITORING (Revolving Line)**\n\nWhile leverage is elevated due to capital returns, cash flows remain elite. Facility limit renewed subject to leverage caps.")
                else:
                    st.success(f"**Underwriting Status: APPROVED**\n\nAutomated review completed for {latest_client['account_name']}.")
            else:
                if tab3_selected_entity == 'TIER_1_PRIME':
                    st.success("**Tier Policy: Prime Automatic Approval**\n\nDefault Probability < 0.5%. Maximum unsecured line limits authorized across prime accounts.")
                elif tab3_selected_entity == 'TIER_2_MID':
                    st.warning("**Tier Policy: Standard Conditional Approval**\n\nDefault Probability 2.0%-4.0%. Requires credit monitoring and secondary collateral buffers.")
                else:
                    st.error("**Tier Policy: High Risk / Restricted**\n\nDefault Probability > 8.0%. Manual underwriting approval and 100% deposit collateral mandatory.")

        st.markdown("---")
        if latest_client['is_financial_institution']:
            st.subheader("💡 Financial Institution Capital Structure & Basel III Stress Test")
            st.caption("Commercial banks are evaluated on regulatory Tier 1 Common Equity (CET1) capital ratios and Liquidity Coverage Ratios (LCR).")
            
            b_col1, b_col2, b_col3 = st.columns(3)
            b_col1.metric("Est. CET1 Capital Ratio", "15.3%", delta="+2.3% above Fed Min (13.0%)")
            b_col2.metric("Liquidity Coverage Ratio (LCR)", "114%", delta="Compliant (>100%)")
            b_col3.metric("Total Deposits / Liabilities Ratio", "0.88x", delta="Stable Funding")

        elif latest_client['account_category'] == 'Corporate':
            st.subheader("💡 Interactive Corporate DCF Valuation & Sensitivity Analysis")
            st.caption("Adjust cost of capital (WACC) and terminal growth rates to evaluate implied enterprise value and sensitivity.")

            col_wacc, col_g = st.columns(2)
            with col_wacc:
                wacc_input = st.slider("Weighted Average Cost of Capital (WACC %)", 4.0, 15.0, 8.5, 0.25)
                wacc = wacc_input / 100.0
            with col_g:
                g_input = st.slider("Terminal Growth Rate (g %)", 0.5, 5.0, 2.5, 0.25)
                g = g_input / 100.0

            tax_rate = 0.21
            base_income = latest_client['operating_income']
            estimated_fcff = base_income * (1 - tax_rate)

            if wacc <= g:
                st.error("⚠️ Cost of Capital (WACC) must be strictly greater than Terminal Growth Rate (g).")
            else:
                terminal_value = (estimated_fcff * (1 + g)) / (wacc - g)
                
                val_kpi1, val_kpi2, val_kpi3 = st.columns(3)
                val_kpi1.metric("Estimated Base FCFF", f'${estimated_fcff / 1e9:,.2f}B')
                val_kpi2.metric("Implied Terminal Value", f'${terminal_value / 1e9:,.2f}B')
                val_kpi3.metric("Valuation Multiplier", f'{(terminal_value / base_income):,.1f}x EBIT' if base_income > 0 else "N/A")

                sens_col1, sens_col2 = st.columns([3, 2])
                
                with sens_col1:
                    wacc_pct_points = [round(wacc_input + delta, 2) for delta in [-3.0, -2.4, -1.8, -1.2, -0.6, 0.0, 0.6, 1.2, 1.8, 2.4, 3.0]]
                    wacc_pct_points = [w for w in wacc_pct_points if w > (g_input + 0.1)]
                    
                    tv_curve = [(estimated_fcff * (1 + g)) / ((w / 100.0) - g) / 1e9 for w in wacc_pct_points]
                    
                    fig_sens = px.line(
                        pd.DataFrame({"WACC (%)": wacc_pct_points, "Terminal Value ($B)": tv_curve}),
                        x="WACC (%)", y="Terminal Value ($B)",
                        title=f"WACC Curve: {latest_client['account_name']} (at g = {g_input:.2f}%)",
                        markers=True
                    )
                    fig_sens.add_scatter(
                        x=[wacc_input], y=[terminal_value / 1e9], 
                        mode="markers+text", marker=dict(color="red", size=14, symbol="diamond"),
                        text=["Active WACC"], textposition="top center", name="Active Selection"
                    )
                    st.plotly_chart(fig_sens, use_container_width=True)

                with sens_col2:
                    st.markdown("**Valuation Sensitivity Matrix ($B):**")
                    wacc_steps = [wacc_input - 1.0, wacc_input, wacc_input + 1.0]
                    g_steps = [g_input - 0.5, g_input, g_input + 0.5]
                    
                    raw_matrix = []
                    for w_val in wacc_steps:
                        row = []
                        w_dec = w_val / 100.0
                        for g_val in g_steps:
                            g_dec = g_val / 100.0
                            if w_dec > g_dec:
                                val = (estimated_fcff * (1 + g_dec)) / (w_dec - g_dec) / 1e9
                                row.append(val)
                            else:
                                row.append(None)
                        raw_matrix.append(row)

                    matrix_df = pd.DataFrame(
                        raw_matrix, 
                        index=[f"WACC {w:.2f}%" for w in wacc_steps],
                        columns=[f"g = {g_val:.2f}%" for g_val in g_steps]
                    )
                    st.dataframe(
                        matrix_df.style.background_gradient(cmap="RdYlGn", axis=None).format(lambda x: f"${x:,.2f}B" if pd.notna(x) else "N/A"),
                        use_container_width=True
                    )

        else:
            st.subheader("💡 Standardized Non-Corporate Tier Stress Test")
            st.caption("Simulate portfolio default rate increases to model required credit loss reserves.")
            
            shock = st.slider("Simulated Default Rate Spike (+%)", 0.0, 10.0, 2.5, 0.5)
            base_loss = latest_client['total_liabilities'] * 0.02
            stressed_loss = latest_client['total_liabilities'] * (0.02 + (shock / 100.0))
            
            t_kpi1, t_kpi2 = st.columns(2)
            t_kpi1.metric("Baseline Reserve Requirement", f"${base_loss / 1e6:,.1f}M")
            t_kpi2.metric("Stressed Loss Reserve", f"${stressed_loss / 1e6:,.1f}M", delta=f"+${(stressed_loss - base_loss) / 1e6:,.1f}M Risk")

with tab4:
    with st.expander("ℹ️ View Methodology & Context (Portfolio Analytics)", expanded=False):
        st.markdown("""
            <div class="context-box">
                <b>📌 Situation:</b> Non-corporate credit tiers require continuous, high-frequency trend monitoring.<br>
                <b>💡 Solution:</b> Monthly granular tracking of D/E leverage ratios alongside transaction density and cash flow trends.<br>
                <b>🔍 What This Means:</b> Allows credit ops to detect monthly cash flow squeezes and tier degradation in real time.
            </div>
        """, unsafe_allow_html=True)

    st.subheader("Non-Corporate Portfolio Risk & Transaction Analytics")

    nc_col1, nc_col2, nc_col3 = st.columns(3)

    with nc_col1:
        fig_dte_tier = px.line(
            df_monthly_tiers,
            x="month",
            y="debt_to_equity",
            color="account_id",
            markers=True,
            title="Monthly Non-Corporate Credit Tier D/E Profile",
            labels={"debt_to_equity": "Tier Leverage (D/E)", "month": "Month", "account_id": "Risk Tier"},
        )
        fig_dte_tier.update_layout(hovermode='x unified')
        st.plotly_chart(fig_dte_tier, use_container_width=True)

    with nc_col2:
        df_monthly_sum = df_transactions.groupby("month")["net_cash_flow"].sum().reset_index()
        fig_cashflow = px.line(
            df_monthly_sum,
            x="month",
            y="net_cash_flow",
            markers=True,
            title="Monthly Net Cash Flow Trend ($)",
            labels={"month": "Month", "net_cash_flow": "Net Cash Flow ($)"},
            line_shape="spline"
        )
        fig_cashflow.update_traces(line_color="#1F4E78", line_width=3)
        st.plotly_chart(fig_cashflow, use_container_width=True)

    with nc_col3:
        fig_volume = px.bar(
            df_transactions,
            x="month",
            y="transaction_count",
            color="account_type",
            title="Monthly Transaction Density",
            labels={"month": "Month", "transaction_count": "Transaction Count", "account_type": "Account Type"},
            barmode="stack"
        )
        st.plotly_chart(fig_volume, use_container_width=True)

with tab5:
    with st.expander("ℹ️ View Methodology & Context (Data Ledger)", expanded=False):
        st.markdown("""
            <div class="context-box">
                <b>📌 Situation:</b> Regulators and auditors demand a single, unaggregated ledger for verification.<br>
                <b>💡 Solution:</b> A standardized credit ledger displaying normalized corporate and tier balances.<br>
                <b>🔍 What This Means:</b> Provides full transparency and direct export capabilities for internal and external audit compliance.
            </div>
        """, unsafe_allow_html=True)

    st.subheader("Standardized Credit Risk Data Ledger")
    st.dataframe(
        filtered_df.style.format(
            {
                "total_assets": "${:,.0f}",
                "total_liabilities": "${:,.0f}",
                "equity": "${:,.0f}",
                "operating_income": "${:,.0f}",
                "debt_to_equity": "{:.2f}x",
                "interest_coverage_ratio": "{:.2f}x",
                "altman_z_score": lambda x: f"{x:.2f}" if pd.notna(x) else "-",
            }
        ),
        use_container_width=True,
    )

Overwriting app.py
